# Get Raw Statistics Canada Basemap

## Purpose

This notebook downloads and organizes the Statistics Canada boundary shapefile required by the Geospatial-CANOE basemap workflow.

No grid generation, CRS transformation, boundary dissolve, adjacency construction, or schema encoding is performed here. The goal is only to acquire the raw boundary shapefile and place it in the expected project directory structure.

---

## Data Source

**Statistics Canada boundary file**

- Source: Statistics Canada
- Format: Shapefile (`.shp`) distributed as a ZIP archive
- Coverage: Canadian provincial and territorial boundaries
- Purpose: Raw national boundary input for constructing latitude–longitude basemap grids.

Source page:

```text
https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21
```

---

## Expected Output Structure

```text
data_files/
└── raw/
    └── basemaps/
        ├── *.shp
        ├── *.shx
        ├── *.dbf
        ├── *.prj
        └── ...
```

Later scripts assume this folder contains exactly one boundary shapefile.

---

## Workflow

This notebook performs the following stages:

1. Create the required raw basemap folder.
2. Define the Statistics Canada boundary download URL.
3. Download the ZIP archive.
4. Extract the shapefile components.
5. Validate that exactly one `.shp` file exists in the raw basemap folder.

In [2]:
# =============================================================================
# Cell 2 — Imports and project paths
# =============================================================================

from pathlib import Path
import shutil
import zipfile
import time

import requests


# Project root
PROJECT_ROOT = Path.cwd().parents[0]

# If running from a different working directory:
# PROJECT_ROOT = Path(r"C:\Users\aviga\Research\repos\temoa_geospace")


# Raw basemap directory
DATA_FILES = PROJECT_ROOT / "data_files"
RAW_DATA = DATA_FILES / "raw"
RAW_BASEMAPS = RAW_DATA / "basemaps"


# Create required directory
RAW_BASEMAPS.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"Project root: {PROJECT_ROOT}")
print(f"Raw basemap folder: {RAW_BASEMAPS}")

Project root: c:\Users\aviga\Research\repos\temoa_geospace
Raw basemap folder: c:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\basemaps


In [4]:
# =============================================================================
# Cell 3 — Download settings
# =============================================================================

HEADERS = {
    "User-Agent": (
        "Geospatial-CANOE/0.1.0 "
        "(University of Toronto Academic Research)"
    )
}

REQUEST_TIMEOUT = 120          # seconds
MAX_RETRIES = 3
DOWNLOAD_DELAY = 2             # seconds between downloads
CHUNK_SIZE = 1024 * 1024       # 1 MB streaming chunks

print("Download settings configured.")

Download settings configured.


In [5]:
# =============================================================================
# Cell 4 — Basemap source definition
# =============================================================================
# Statistics Canada provincial and territorial boundary file
#
# Source page:
# https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index-eng.cfm
#
# Direct download:
# https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lpr_000b21a_e.zip
#
# This cell only defines the basemap source. It does not download anything.

BASEMAP_SOURCE_PAGE = (
    "https://www12.statcan.gc.ca/census-recensement/2021/geo/"
    "sip-pis/boundary-limites/index-eng.cfm"
)

BASEMAP_URL = (
    "https://www12.statcan.gc.ca/census-recensement/2021/geo/"
    "sip-pis/boundary-limites/files-fichiers/lpr_000b21a_e.zip"
)

BASEMAP_RESOURCE = {
    "name": "Statistics Canada 2021 provincial and territorial boundary file",
    "url": BASEMAP_URL,
    "output_dir": RAW_BASEMAPS,
    "archive_name": "lpr_000b21a_e.zip",
    "expected_shapefile": "lpr_000b21a_e.shp",
}

print("Basemap source configured")
print(f"Source page: {BASEMAP_SOURCE_PAGE}")
print(f"Download URL: {BASEMAP_RESOURCE['url']}")
print(f"Output folder: {BASEMAP_RESOURCE['output_dir']}")
print(f"Expected shapefile: {BASEMAP_RESOURCE['expected_shapefile']}")

Basemap source configured
Source page: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index-eng.cfm
Download URL: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/files-fichiers/lpr_000b21a_e.zip
Output folder: c:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\basemaps
Expected shapefile: lpr_000b21a_e.shp


In [6]:
# =============================================================================
# Cell 5 — Download and extraction helper functions
# =============================================================================

def download_file(url, destination):
    """
    Download a file from a URL if it does not already exist.
    """

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        print(f"[Skip download] {destination.name} already exists.")
        return destination

    print(f"[Download] {destination.name}")

    response = requests.get(
        url,
        headers=HEADERS,
        stream=True,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:
                f.write(chunk)

    print(f"[Complete download] {destination.name}")

    time.sleep(DOWNLOAD_DELAY)

    return destination


def extract_basemap_archive(zip_path, output_dir, overwrite=False):
    """
    Extract the Statistics Canada basemap ZIP archive into the raw basemap folder.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    existing_shapefiles = sorted(output_dir.glob("*.shp"))

    if existing_shapefiles and not overwrite:
        print(f"[Skip extract] {output_dir.name} already contains shapefile(s).")
        return existing_shapefiles

    print(f"[Extract] {zip_path.name} → {output_dir.name}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)

    shapefiles = sorted(output_dir.rglob("*.shp"))

    if not shapefiles:
        raise FileNotFoundError(
            f"No shapefile found after extracting {zip_path.name}"
        )

    for source_path in shapefiles:
        destination_path = output_dir / source_path.name

        if source_path.parent == output_dir:
            continue

        if destination_path.exists():
            if overwrite:
                destination_path.unlink()
            else:
                print(f"[Skip move] {destination_path.name} already exists.")
                continue

        shutil.move(str(source_path), str(destination_path))
        print(f"[Move] {source_path.name} → {destination_path.name}")

    for path in sorted(output_dir.iterdir()):
        if path.is_dir():
            shutil.rmtree(path)

    final_shapefiles = sorted(output_dir.glob("*.shp"))

    if len(final_shapefiles) != 1:
        raise ValueError(
            f"Expected exactly 1 shapefile in {output_dir}, "
            f"found {len(final_shapefiles)}"
        )

    if zip_path.exists():
        zip_path.unlink()
        print(f"[Delete] {zip_path.name}")

    print(f"[Complete] {output_dir.name}: {final_shapefiles[0].name}")

    return final_shapefiles

In [7]:
# =============================================================================
# Cell 6 — Download and extract basemap archive
# =============================================================================

archive_path = RAW_BASEMAPS / BASEMAP_RESOURCE["archive_name"]

downloaded_archive = download_file(
    url=BASEMAP_RESOURCE["url"],
    destination=archive_path,
)

basemap_files = extract_basemap_archive(
    zip_path=downloaded_archive,
    output_dir=BASEMAP_RESOURCE["output_dir"],
    overwrite=False,
)

print("\nBasemap acquisition summary")
print("---------------------------")

for file in sorted(RAW_BASEMAPS.iterdir()):
    if file.is_file():
        print(f"- {file.name}")

print(f"\nShapefile ready: {basemap_files[0]}")

[Download] lpr_000b21a_e.zip
[Complete download] lpr_000b21a_e.zip
[Extract] lpr_000b21a_e.zip → basemaps
[Delete] lpr_000b21a_e.zip
[Complete] basemaps: lpr_000b21a_e.shp

Basemap acquisition summary
---------------------------
- lpr_000b21a_e.dbf
- lpr_000b21a_e.prj
- lpr_000b21a_e.shp
- lpr_000b21a_e.shx
- lpr_000b21a_e.xml

Shapefile ready: c:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\basemaps\lpr_000b21a_e.shp
